In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold

In [ ]:
df = pd.read_csv("Dataset/Harga Bahan Pangan/train/Daging Ayam Ras.csv")

In [3]:
def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [4]:
df.head()

,Date,Aceh,Bali,Banten,Bengkulu,DI Yogyakarta,DKI Jakarta,Gorontalo,Jambi,Jawa Barat,...,Papua,Riau,Sulawesi Barat,Sulawesi Selatan,Sulawesi Tengah,Sulawesi Tenggara,Sulawesi Utara,Sumatera Barat,Sumatera Selatan,Sumatera Utara
0,2022-01-01,30540.0,38390.0,37610.0,39680.0,36180.0,35990.0,31710.0,37210.0,37640.0,...,44810.0,34770.0,30880.0,26460.0,39410.0,33530.0,36550.0,29780.0,35470.0,36090.0
1,2022-01-02,31390.0,38340.0,36440.0,38820.0,35970.0,36480.0,31200.0,36680.0,37350.0,...,43750.0,35100.0,32290.0,26130.0,38640.0,32500.0,35480.0,31230.0,35410.0,36670.0
2,2022-01-03,30980.0,38660.0,37440.0,39740.0,35590.0,37160.0,31490.0,37690.0,37080.0,...,43910.0,35030.0,31900.0,26270.0,34620.0,34580.0,35980.0,31780.0,35680.0,35790.0
3,2022-01-04,31680.0,38660.0,37320.0,40960.0,35590.0,36180.0,31320.0,37700.0,37040.0,...,43750.0,37220.0,31560.0,26200.0,38840.0,34840.0,35710.0,31180.0,36010.0,36540.0
4,2022-01-05,32620.0,38340.0,36900.0,40960.0,35480.0,36370.0,31250.0,37720.0,36690.0,...,42830.0,37010.0,31220.0,26240.0,38410.0,34540.0,35470.0,32110.0,34430.0,35860.0


In [5]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                         0.000000
Aceh                         3.685259
Bali                         3.585657
Banten                       3.685259
Bengkulu                     3.685259
DI Yogyakarta                3.585657
DKI Jakarta                  3.685259
Gorontalo                    3.486056
Jambi                        3.784861
Jawa Barat                   3.685259
Jawa Tengah                  3.386454
Jawa Timur                   3.486056
Kalimantan Barat             3.585657
Kalimantan Selatan           3.685259
Kalimantan Tengah            3.585657
Kalimantan Timur             3.884462
Kalimantan Utara             3.884462
Kepulauan Bangka Belitung    3.784861
Kepulauan Riau               3.884462
Lampung                      3.685259
Maluku Utara                 3.585657
Maluku                       3.685259
Nusa Tenggara Barat          3.685259
Nusa Tenggara Timur          3.386454
Papua Barat                  3.884462
Papua                        3.685259
Riau        

In [6]:
numeric_features = df.select_dtypes(include=['number']).columns
df[numeric_features] = df[numeric_features].fillna(df[numeric_features].median())

In [7]:
zero_var_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Columns with zero variance:", zero_var_cols)

Columns with zero variance: []


In [8]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                         0.0
Aceh                         0.0
Bali                         0.0
Banten                       0.0
Bengkulu                     0.0
DI Yogyakarta                0.0
DKI Jakarta                  0.0
Gorontalo                    0.0
Jambi                        0.0
Jawa Barat                   0.0
Jawa Tengah                  0.0
Jawa Timur                   0.0
Kalimantan Barat             0.0
Kalimantan Selatan           0.0
Kalimantan Tengah            0.0
Kalimantan Timur             0.0
Kalimantan Utara             0.0
Kepulauan Bangka Belitung    0.0
Kepulauan Riau               0.0
Lampung                      0.0
Maluku Utara                 0.0
Maluku                       0.0
Nusa Tenggara Barat          0.0
Nusa Tenggara Timur          0.0
Papua Barat                  0.0
Papua                        0.0
Riau                         0.0
Sulawesi Barat               0.0
Sulawesi Selatan             0.0
Sulawesi Tengah              0.0
Sulawesi T

In [9]:
numerical_features = df.select_dtypes(include=['number'])
categorical_features = df.select_dtypes(exclude=['number'])

# Apply VarianceThreshold to remove low-variance numerical features
selector = VarianceThreshold(threshold=0.01)  # Adjust threshold as needed
reduced_numerical_df = selector.fit_transform(numerical_features)

# Convert back to DataFrame with selected features
reduced_numerical_df = pd.DataFrame(reduced_numerical_df, 
                                       columns=numerical_features.columns[selector.get_support()])

# Combine numerical and categorical features back together
reduced_df = pd.concat([reduced_numerical_df, categorical_features.reset_index(drop=True)], axis=1)

In [11]:
df.shape

(1004, 35)

In [12]:
reduced_df.shape

(1004, 35)

In [13]:
skewness = df.select_dtypes(include=['number']).apply(lambda x: stats.skew(x.dropna())).sort_values(ascending=False)
print(skewness)

Sulawesi Selatan             1.046790
Sulawesi Tengah              1.008642
Kalimantan Tengah            0.979496
Sulawesi Barat               0.931704
Jambi                        0.913838
Jawa Barat                   0.874312
Kalimantan Selatan           0.871151
Jawa Tengah                  0.778796
Banten                       0.707855
Sumatera Barat               0.701304
Bengkulu                     0.695052
Kalimantan Barat             0.655230
Aceh                         0.650549
Riau                         0.642715
Sumatera Utara               0.626508
Kepulauan Bangka Belitung    0.581653
Sulawesi Utara               0.564338
Sumatera Selatan             0.558184
DI Yogyakarta                0.508572
Lampung                      0.495262
Jawa Timur                   0.458747
Nusa Tenggara Timur          0.398887
Sulawesi Tenggara            0.300554
DKI Jakarta                  0.287368
Maluku                       0.260084
Kepulauan Riau               0.215048
Papua       

In [14]:
# Select numerical columns
num_cols = df.select_dtypes(include=['number'])

# Function to calculate outlier percentage using IQR
def outlier_percentage(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((column < lower_bound) | (column > upper_bound)).sum()
    return (outliers / len(column)) * 100  # Percentage

# Apply function to all numerical columns
outlier_percentages_df = num_cols.apply(outlier_percentage)

# Display the results
print(outlier_percentages_df.sort_values(ascending=False))

Kalimantan Tengah            6.374502
Sulawesi Tengah              5.976096
Sulawesi Selatan             4.083665
Maluku Utara                 3.386454
Sulawesi Barat               2.888446
Kalimantan Selatan           2.788845
Kepulauan Riau               2.589641
Nusa Tenggara Barat          2.290837
Jawa Barat                   1.992032
Gorontalo                    1.892430
DKI Jakarta                  1.792829
Banten                       1.693227
Papua Barat                  1.693227
Jambi                        1.394422
Kalimantan Barat             1.294821
Lampung                      1.195219
Jawa Tengah                  1.095618
Sulawesi Utara               0.896414
Maluku                       0.896414
Sulawesi Tenggara            0.796813
DI Yogyakarta                0.697211
Sumatera Selatan             0.697211
Nusa Tenggara Timur          0.597610
Papua                        0.498008
Aceh                         0.398406
Kepulauan Bangka Belitung    0.398406
Kalimantan U

In [15]:
test = pd.read_csv("Dataset/Harga Bahan Pangan/test/Daging Ayam Ras.csv")

In [ ]:
df.to_csv("Daging Ayam Ras.csv", index=False)

In [16]:
def df_to_X_y(df, window_size=5):
  df_as_np = df.to_numpy()
  X = []
  y = []
  for i in range(len(df_as_np)-window_size):
    row = [[a] for a in df_as_np[i:i+window_size]]
    X.append(row)
    label = df_as_np[i+window_size]
    y.append(label)
  return np.array(X), np.array(y)

In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping


In [18]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer, Dropout, Conv1D, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split

# Load dataset
df = df.select_dtypes(include=[np.number])  # Keep only numeric columns
df = df.apply(pd.to_numeric, errors='coerce').dropna()  # Convert and drop NaNs

# If multiple numeric columns exist, use the first one
if df.shape[1] > 1:
    print(f"Warning: DataFrame has multiple numeric columns ({df.shape[1]}). Using the first column.")
    df = df.iloc[:, 0]

# Convert data into sequences
def df_to_X_y(df, window_size=10):  # Keep window size = 10 for CNN effectiveness
    X, y = [], []
    for i in range(len(df) - window_size):
        X.append(df[i:i+window_size].values)  # Directly use raw values
        y.append(df.iloc[i + window_size])  # Keep original values
    return np.array(X), np.array(y)

X, y = df_to_X_y(df, window_size=10)

# Reshape for CNN-LSTM (Conv1D requires 3D input)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# Callbacks
checkpoint_path = "model_checkpoint.keras"
cp4 = ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_loss', mode='min')
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.8, patience=5, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# Define CNN-LSTM model
def create_cnn_lstm_model(input_shape):
    model = Sequential([
        InputLayer(input_shape=input_shape),

        # CNN Layers
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),

        # LSTM Layers
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.3),

        # Dense Layers
        Dense(16, activation='relu'),
        Dense(1, activation='linear')  # Output remains in original scale
    ])
    
    model.compile(loss='mape', optimizer=Adam(learning_rate=0.003), metrics=['mape'])
    return model

# Train the model
input_shape = (X_train.shape[1], X_train.shape[2])
model = create_cnn_lstm_model(input_shape)

history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
          epochs=300, batch_size=32, verbose=1, 
          callbacks=[cp4, lr_reducer, early_stopping])

print("Training completed. Final epoch:", len(history.history['loss']))

# Save the model
model.save("cnn_lstm_model.keras")

# Forecasting
df_submission = pd.read_csv("Dataset/Harga Bahan Pangan/sample_submission.csv")
unique_countries = df_submission['id'].str.split('/').str[1].unique()
total_required_predictions = 92 * len(unique_countries)

future_predictions = []
input_seq = X_test[-1]

for _ in range(total_required_predictions):
    pred = model.predict(input_seq.reshape(1, input_seq.shape[0], 1))[0, 0]
    pred += np.random.normal(0, 0.01)  # Add small noise to prevent stagnation
    future_predictions.append(pred)
    input_seq = np.roll(input_seq, -1)
    input_seq[-1] = pred

# Check if predictions match expected count
if len(future_predictions) != total_required_predictions:
    print(f"Warning: Expected {total_required_predictions} predictions but got {len(future_predictions)}")

# Save submission file
data = [{'id': df_submission.iloc[i]['id'], 'price': future_predictions[i]} for i in range(min(len(df_submission), len(future_predictions)))]
submission_df = pd.DataFrame(data)
submission_df.to_csv("Daging_Ayam_Ras_cnn_lstm_submission.csv", index=False)
print("Submission file saved as Daging_Ayam_Ras_cnn_lstm_submission.csv")

d:\Anaconda\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning:

Argument `input_shape` is deprecated. Use `shape` instead.



Epoch 1/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - loss: 99.9951 - mape: 99.9951 - val_loss: 99.9711 - val_mape: 99.9711 - learning_rate: 0.0030
Epoch 2/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 99.9620 - mape: 99.9620 - val_loss: 99.9311 - val_mape: 99.9311 - learning_rate: 0.0030
Epoch 3/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 99.9182 - mape: 99.9182 - val_loss: 99.8778 - val_mape: 99.8778 - learning_rate: 0.0030
Epoch 4/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 99.8614 - mape: 99.8614 - val_loss: 99.8089 - val_mape: 99.8089 - learning_rate: 0.0030
Epoch 5/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 99.7859 - mape: 99.7859 - val_loss: 99.7227 - val_mape: 99.7227 - learning_rate: 0.0030
Epoch 6/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 99.6915 - mape: 99.6915 - val_loss: 99.6150 - val_mape: 99.6150 - learning_rate: 0.0030
Epoch 7/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 99.5847 - mape: 99.5847 - val_loss: 99.4897 - val_ma